# Optimize All Dataset Configs

Run CLASP latent Bayesian optimization for every configured dataset and save the best preprocessing, estimator, and graph settings to `data/optimized_params/`. Notebook 02 reads these files with `load_or_default_params(...)`, so this notebook provides the shared optimization baseline for visualization and evaluation.

Run `00_download_datasets.ipynb` first so the configured `.h5ad` inputs exist locally. The loop can be resumed: datasets with existing optimized parameter files are skipped unless `overwrite_existing = True`.

In [1]:
from __future__ import annotations

import os

from clasp.notebook_utils import (
    PAPER_DATASET_DOWNLOADS,
    resolve_project_root,
    run_dataset_optimization_sweep,
)


In [2]:
project_root = resolve_project_root()
os.chdir(project_root)

random_state = 0
selected_datasets = list(PAPER_DATASET_DOWNLOADS)

skip_missing_inputs = True
skip_existing_optimized = True
overwrite_existing = False
run_gplvm_refinement = True

embedding_method = "umap"
embedding_epochs = 60
invalid_score = -1e9
summary_path = project_root / "data" / "optimized_params" / "all_datasets_optimization_summary.csv"

base_n_top_genes = 1500
fixed_preprocess_params = {"max_cells": 1000}
base_estimator_params = {"n_components": 60}
base_graph_params = {
    "n_neighbors": 15,
    "intra_fraction": 0.5,
    "n_inter_edges": 2,
    "metric": "euclidean",
    "assignment_quantile": 0.8,
    "hubness_correction": "csls",
    "hubness_k": 10,
    "rank_correction": True,
    "edge_weighting": "binary",
    "mutual_neighbors": False,
    "neighbor_mode": "distance",
    "symmetrize": True,
}

preprocess_search_space = {
    "n_top_genes": {"type": "int", "bounds": [500, 2000]},
}
estimator_search_space = {
    "n_components": {"type": "int", "bounds": [20, 100]},
}
graph_search_space = {
    "n_neighbors": {"type": "int", "bounds": [5, 35]},
    "intra_fraction": {"type": "float", "bounds": [0.2, 0.9]},
    "n_inter_edges": {"type": "int", "bounds": [1, 6]},
    "assignment_quantile": {"type": "float", "bounds": [0.1, 1.0]},
    "hubness_k": {"type": "int", "bounds": [3, 20]},
    "rank_correction": {"type": "categorical", "values": [False, True]},
    "edge_weighting": {"type": "categorical", "values": ["binary", "distance"]},
    "inter_edge_mode": {"type": "categorical", "values": ["propagate_neighbors", "assignment"]},
    "mutual_neighbors": {"type": "categorical", "values": [False, True]},
}

compact_radii = {
    "n_top_genes": 300,
    "n_components": 20,
    "n_neighbors": 6,
    "intra_fraction": 0.1,
    "n_inter_edges": 2,
    "assignment_quantile": 0.15,
    "hubness_k": 4,
}

pca_bo_settings = {
    "n_initial": 6,
    "latent_dim": 3,
    "n_iterations": 12,
    "embedding_model": "pca",
    "acquisition": "ei",
    "batch_size": 1,
}

gplvm_bo_settings = {
    "n_initial": 6,
    "latent_dim": 3,
    "n_iterations": 12,
    "embedding_model": "gplvm",
    "acquisition": "ei",
    "batch_size": 1,
}

selected_datasets


['scib_pancreas',
 'scib_lung_atlas',
 'scib_immune_human',
 'scib_immune_human_mouse',
 'cellrank_bone_marrow',
 'cellrank_lung',
 'cellrank_pancreas',
 'cellrank_reprogramming_schiebinger',
 'cellrank_reprogramming_morris',
 'zebrafish']

In [ ]:
summary = run_dataset_optimization_sweep(
    selected_datasets,
    project_root=project_root,
    random_state=random_state,
    skip_missing_inputs=skip_missing_inputs,
    skip_existing_optimized=skip_existing_optimized,
    overwrite_existing=overwrite_existing,
    run_gplvm_refinement=run_gplvm_refinement,
    embedding_method=embedding_method,
    embedding_epochs=embedding_epochs,
    invalid_score=invalid_score,
    summary_path=summary_path,
    fixed_preprocess_params=fixed_preprocess_params,
    base_n_top_genes=base_n_top_genes,
    base_estimator_params=base_estimator_params,
    base_graph_params=base_graph_params,
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
    compact_radii=compact_radii,
    pca_bo_settings=pca_bo_settings,
    gplvm_bo_settings=gplvm_bo_settings,
    display_fn=display,
)

summary


[1/10] scib_pancreas
Existing optimized parameter file at /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/clasp/data/optimized_params/scib_pancreas_graph_params.json lacks matching dataset metadata; rerunning optimization.


The JSON parameter files written by this run are consumed directly by notebook 02:

```python
optimized_params = load_or_default_params(selected_dataset, dataset)
```

The CSV summary is only a run log; the JSON files in `data/optimized_params/` are the source of truth for downstream visualization and evaluation.